# ⚡ Advanced SQL for Data Engineers — Interactive Notebook

**Author:** Youssef Ibrahim Mohamed Soliman  
**GitHub:** https://github.com/Yosef-Ibrahim  
**Email:** youssefibrahimelisely@gmail.com  
**Phone:** 01119834356  

---  
This notebook demonstrates Advanced SQL queries (Joins, Window Functions, CTEs, Self Joins, Non-Equi Joins) running against an in-memory SQLite database.

In [ ]:
import sqlite3
import pandas as pd

# Connect to in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Set up schema and sample data
cursor.executescript('''
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE employees (
    emp_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    salary REAL NOT NULL,
    dept_id INTEGER,
    supervisorID TEXT
);

CREATE TABLE job_grades (
    grade TEXT PRIMARY KEY,
    lowsal REAL,
    highsal REAL
);

INSERT INTO departments VALUES (100, 'IT'), (200, 'Marketing'), (300, 'HR');

INSERT INTO job_grades VALUES 
('C', 1000.0, 3500.0), 
('B', 3500.01, 7000.0), 
('A', 7000.01, 15000.0);

INSERT INTO employees VALUES
('1122', 'Ahmed Ali', 5000.0, 100, '2233'),
('2233', 'Kamel Mohamed', 8500.0, 100, '1234'),
('1234', 'Hanaa Sobhy', 9000.0, 200, '2233'),
('3216', 'Amr Omran', 3000.0, 100, NULL),
('9685', 'Noha Mohamed', 4200.0, 200, '2233');
''')
conn.commit()
print('Schema & sample data initialized successfully!')

## 1. Advanced Joins (Equi, Non-Equi & Self Join)

In [ ]:
# Equi-Join Query
pd.read_sql_query('''
SELECT e.emp_id, e.name AS employee_name, d.name AS dept_name, e.salary
FROM employees e
JOIN departments d ON e.dept_id = d.id;
''', conn)

In [ ]:
# Non-Equi Join (Salary Grade Lookup)
pd.read_sql_query('''
SELECT e.name, e.salary, j.grade
FROM employees e
JOIN job_grades j ON e.salary BETWEEN j.lowsal AND j.highsal;
''', conn)

## 2. Window Functions & CTEs

In [ ]:
# Window Function: Department Average Salary & Salary Rank
pd.read_sql_query('''
SELECT name, salary, dept_id,
       AVG(salary) OVER (PARTITION BY dept_id) AS dept_avg_salary,
       DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dept_rank
FROM employees;
''', conn)

In [ ]:
# Common Table Expression (CTE)
pd.read_sql_query('''
WITH HighEarners AS (
    SELECT emp_id, name, salary, dept_id
    FROM employees
    WHERE salary >= 5000
)
SELECT d.name AS department, COUNT(h.emp_id) AS high_earner_count
FROM HighEarners h
JOIN departments d ON h.dept_id = d.id
GROUP BY d.name;
''', conn)